# Photo d'une page anglaise → table de vocabulaire

1. **OCR** (Tesseract, `image_to_data`) : chaque mot avec sa boîte et sa confiance, mots coupés recollés.
2. **Choix du mot** : par son texte (`find`) ou par un clic dans l'image (`word_at`).
3. **Dictionnaire local** (Wiktionnaire via kaikki.org) : lemme (`ran` → `run`), sens, traductions FR.
4. **Synonymes** : Wiktionnaire + WordNet.
5. **Table de vocabulaire** (SQLite) avec export CSV pour Anki.

Tout est gratuit et hors ligne. Prérequis, une seule fois : `make build && make dict`.

In [ ]:
from vocab import ocr_words, annotate, find, sentence_of, page_text, Dictionary, VocabTable

IMAGE = "example_en.jpg"   # remplace par ta photo, en pleine résolution de préférence
words = ocr_words(IMAGE, lang="eng")
print(page_text(words))

In [ ]:
# Boîtes : vert = OCR sûr, orange = moyen, rouge = douteux
annotate(IMAGE, words)

In [ ]:
dico = Dictionary()
table = VocabTable()

choix = find(words, "ran")[0]          # ou word_at(words, x, y) pour un clic
phrase = sentence_of(words, choix)
entrees = dico.lookup(choix.full)

print(f"« {choix.full} » dans : {phrase}\n")
for i, e in enumerate(entrees):
    print(f"[{i}] {e.word} ({e.pos}){'  ← ' + e.via_form if e.via_form else ''}")
    for sens, fr in e.fr.items():
        print(f"     • {sens or '(sens non précisé)'} → {', '.join(fr)}")
    if not e.fr:
        for s in e.senses[:5]:
            print(f"     • {s.gloss}")
    print(f"     synonymes : {', '.join(e.synonyms[:10])}")
annotate(IMAGE, words, highlight=choix)

In [ ]:
# Choisis l'entrée et le sens qui correspondent à la phrase, puis ajoute à la table
e = entrees[0]
sens = "to manage"                      # une des clés de e.fr affichées plus haut
table.add(word=choix.full, lemma=e.word, pos=e.pos, french=e.fr[sens],
          synonyms=e.synonyms[:6], definition=sens, sentence=phrase, source=IMAGE)

In [ ]:
import pandas as pd  # pour un affichage en tableau
pd.DataFrame(table.all())[["lemma", "pos", "french", "synonyms", "sentence", "times_seen"]]

In [ ]:
# Export pour Anki : Fichier → Importer → data/anki.csv (séparateur virgule, HTML autorisé)
table.export_anki("data/anki.csv")